# Executive Widget Dashboard

Interactive country and industry filters with KPI cards and a revenue/profit view

In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

df = pd.read_csv("data/software_companies_dataset_v2.csv")

# Prepare missing values for reliable widgets and Plotly charts.
categorical_columns = [
    "Company_Name", "Industry", "Headquarters_City", "Country",
    "Ownership_Type", "Customer_Segment", "Primary_Cloud", "Risk_Rating"
]
for column in categorical_columns:
    if column in df.columns:
        df[column] = df[column].fillna("Unknown").astype(str).str.strip()

numeric_columns = [
    "Employees", "Annual_Revenue", "Profit_Margin", "Market_Share",
    "R&D_Spending", "Average_Salary", "Training_Hours_Per_Employee",
    "Employee_Satisfaction", "Adoption_Rate_AI", "Adoption_Rate_Cloud",
    "Adoption_Rate_Blockchain"
]
for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")
        df[column] = df[column].fillna(df[column].median())

df["Annual_Revenue"] = df["Annual_Revenue"].clip(lower=1)
df["Employees"] = df["Employees"].clip(lower=1)

country = widgets.SelectMultiple(options=sorted(df["Country"].dropna().astype(str).unique().tolist()), description="Country")
industry = widgets.SelectMultiple(options=sorted(df["Industry"].dropna().astype(str).unique().tolist()), description="Industry")
out = widgets.Output()

def render(*_):
    view = df.copy()
    if country.value: view = view[view["Country"].isin(country.value)]
    if industry.value: view = view[view["Industry"].isin(industry.value)]
    with out:
        clear_output(wait=True)
        display(widgets.HTML(
            f"<h3>Companies: {len(view):,} | Revenue: ${view.Annual_Revenue.sum()/1e9:,.2f}B "
            f"| Employees: {view.Employees.sum():,.0f}</h3>"
        ))
        agg = view.groupby("Industry", as_index=False).agg(
            Revenue=("Annual_Revenue","sum"), Margin=("Profit_Margin","mean"))
        fig = px.bar(agg, x="Industry", y="Revenue", color="Margin",
                     title="Revenue and average margin by industry")
        fig.show()

country.observe(render, names="value")
industry.observe(render, names="value")
display(widgets.HBox([country, industry]), out)
render()